In [1]:
%load_ext autoreload
%autoreload 2

# Load the BPMN Dataset

In [2]:
from mcp4cm.bpmn.dataloading import BPMNDataset, load_dataset_from_csv
from mcp4cm.dataloading import load_dataset
from mcp4cm.base import DatasetType


In [3]:
bpmn_dataset = load_dataset(dataset_type=DatasetType.BPMNMODELSET, path='data/bpmnmodelset')


Loading SAP SAM Dataset @ data/bpmnmodelset\sap_sam_2022/models:   0%|          | 0/103 [00:00<?, ?it/s]

Defining file path for saving dataset and loading it later on.

In [4]:
file_path = 'data/bpmnmodelset/processed/culled_models.csv'


In [5]:
from mcp4cm.bpmn.data_extraction import extract_names_from_models

use_types = False

if use_types:
    key = 'names_with_types'
    file_path = 'data/bpmnmodelset/processed/culled_with_typed_names.csv'
    empty_name = 'unknown type: empty name'
else:
    key = 'names'
    file_path = 'data/bpmnmodelset/processed/culled_with_names.csv'
    empty_name = 'empty name'



extract_names_from_models(bpmn_dataset, use_types=use_types)


Extracting names from raw model done.


In [ ]:
len(bpmn_dataset)

In [6]:
from mcp4cm.bpmn.data_extraction import (filter_empty_models,
                                         filter_models_by_element_count,
                                         filter_models_by_empty_name_percentage,
                                         filter_models_by_dummy_words)
from mcp4cm.dataloading import Dataset

print(bpmn_dataset)
Dataset.apply_filters(
    dataset=bpmn_dataset,
    filters=[
        filter_empty_models,
        filter_models_by_element_count,
        filter_models_by_empty_name_percentage,
        filter_models_by_dummy_words
    ],
)

print(bpmn_dataset)

Dataset(name=sapsam_2022_bpmn2, models=30312)
Found models with empty names: 808
Filtered out models with element counts outside of 5 and 200: 408
Filtered out models with a empty name percentage higher than 0.5: 14757
Filtered out models with a dummy_percentage higher than 0.6: 4
Dataset(name=sapsam_2022_bpmn2, models=14335)


In [ ]:
from mcp4cm.generic.language_detection import detect_dataset_languages
from mcp4cm.bpmn.data_extraction import extract_model_languages

extract_model_languages(bpmn_dataset, key=key, empty_name=empty_name)


In [ ]:
language_dict = detect_dataset_languages(bpmn_dataset)

In [ ]:
from mcp4cm.generic.language_detection import filter_models_by_language
english_dataset = filter_models_by_language(bpmn_dataset, 'en', key=key, empty_name=empty_name)
language_dict = detect_dataset_languages(english_dataset)

file_path = 'data/bpmnmodelset/processed/english_models.csv'
print(len(english_dataset))
BPMNDataset.to_csv(english_dataset, file_path);


In [ ]:
del(english_dataset)

In [ ]:
from mcp4cm.bpmn.dataloading import load_dataset_from_csv

file_path = 'data/bpmnmodelset/processed/english_models.csv'
loaded_dataset = load_dataset_from_csv('english_bpmn', fp=file_path)

In [ ]:
len(loaded_dataset)

In [ ]:
from mcp4cm.generic.duplicate_detection import detect_duplicates_by_hash
detect_duplicates_by_hash(loaded_dataset, inplace=False, plt_fig=True, print_results=True)


In [ ]:
from mcp4cm.generic.duplicate_detection import tfidf_near_duplicate_detector
key = 'names'
_, duplicate_dataset = tfidf_near_duplicate_detector(loaded_dataset,key=key, threshold=0.95, inplace=False, plt_fig=True, print_results=True)

In [ ]:
file_path = 'data/bpmnmodelset/processed/english_deduplicated_models.csv'
BPMNDataset.to_csv(loaded_dataset, file_path);

In [ ]:
duplicate_file_path = 'data/bpmnmodelset/processed/duplicate_models.csv'
BPMNDataset.to_csv(duplicate_dataset, duplicate_file_path);

In [ ]:
duplicate_dataset.models.value_counts('duplicate_group')

In [ ]:
print(duplicate_dataset.models.query('duplicate_group == 4').iloc[0].names)
print(duplicate_dataset.models.query('duplicate_group == 2').iloc[1254].names)